# ID3 *(Iterative Dichotomiser 3)*

### Inducción de árboles de decisión para clasificación supervisada

## 1. Descripción del algoritmo

El algoritmo **ID3** *(Iterative Dichotomiser 3)*, propuesto por J. Ross Quinlan, es un método de **aprendizaje supervisado** para la construcción de **árboles de decisión** orientados fundamentalmente a tareas de **clasificación**.

Su principio rector consiste en particionar recursivamente el conjunto de entrenamiento seleccionando, en cada nodo, el atributo que produce la mayor reducción de la incertidumbre sobre la variable objetivo. Esta reducción se cuantifica mediante dos magnitudes procedentes de la teoría de la información: la **entropía**, que mide la impureza de un conjunto de ejemplos, y la **ganancia de información** *(information gain)*, que evalúa cuánto disminuye dicha impureza al dividir según un atributo.

El procedimiento crea una rama por cada valor del atributo elegido y se repite sobre cada subconjunto resultante hasta que los nodos sean puros, se agoten los atributos disponibles o se satisfaga un criterio de parada. El resultado es un modelo jerárquico e interpretable, expresable como un conjunto de reglas del tipo *si–entonces*.

ID3 constituye el fundamento histórico de algoritmos posteriores más sofisticados, como **C4.5** y **C5.0**, que incorporan poda, manejo de atributos numéricos y tratamiento de valores faltantes.

## 2. Publicación que propuso el algoritmo

### BibTeX
```bibtex
@article{quinlan1986induction,
  title     = {Induction of Decision Trees},
  author    = {Quinlan, J. Ross},
  journal   = {Machine Learning},
  volume    = {1},
  number    = {1},
  pages     = {81--106},
  year      = {1986},
  publisher = {Springer},
  doi       = {10.1007/BF00116251}
}
```

### Referencia APA
Quinlan, J. R. (1986). *Induction of decision trees*. Machine Learning, 1(1), 81–106. https://doi.org/10.1007/BF00116251

## 3. Tipo de modelo

| Dimensión | Clasificación para ID3 |
|---|---|
| **Método de aprendizaje** | Supervisado |
| **Tipo de problema** | Clasificación |
| **Por parámetros** | No paramétrico |
| **Datos al entrenar** | Offline / batch |
| **Resultado del entrenamiento** | Basado en modelo: árbol de decisión |
| **Familia de algoritmos** | Decision Trees |

## 4. Algoritmo de entrenamiento

ID3 usa el criterio de **ganancia de información**.

### Entropía
La entropía mide la impureza o incertidumbre de un conjunto de ejemplos:

$$
H(S) = -\sum_{i=1}^{c} p_i \log_2(p_i)
$$

Donde $p_i$ es la proporción de ejemplos de la clase $i$.

### Ganancia de información
La ganancia de información mide cuánto disminuye la entropía al dividir por un atributo $A$:

$$
IG(S, A) = H(S) - \sum_{v \in Values(A)} \frac{|S_v|}{|S|} H(S_v)
$$

### Proceso general
1. Calcular la entropía del conjunto actual.
2. Calcular la ganancia de información de cada atributo.
3. Seleccionar el atributo con mayor ganancia.
4. Crear una rama por cada valor del atributo.
5. Repetir recursivamente sobre cada subconjunto.
6. Crear una hoja cuando todos los ejemplos tengan la misma clase o cuando ya no existan atributos útiles.

## 5. Tipos de entrada

- Datos **tabulares** estructurados.
- Variables explicativas normalmente **categóricas**.
- Variable objetivo **categórica**.
- Para variables numéricas se recomienda discretización previa si se desea una implementación ID3 pura.
- Para valores faltantes se recomienda imputación o una estrategia explícita, porque ID3 básico no maneja missing values de forma robusta.

## 6. Casos de uso

| Área | Ejemplo |
|---|---|
| Medicina | Clasificar si un paciente tiene riesgo alto/bajo según síntomas |
| Finanzas | Aprobar o rechazar un crédito |
| Marketing | Clasificar clientes según probabilidad de compra |
| Educación | Predecir si un estudiante aprobará o reprobará |
| Operaciones | Clasificar incidentes o tickets por prioridad |
| Sistemas expertos | Generar reglas IF-THEN interpretables |

## 7. Supuestos y restricciones

### Supuestos
- Los datos de entrenamiento son representativos del problema real.
- Las variables explicativas contienen información útil para separar las clases.
- La relación entre variables y clase puede expresarse mediante particiones jerárquicas.

### Restricciones
- Tiende al **sobreajuste** si el árbol crece sin control.
- Favorece atributos con muchos valores posibles, porque pueden producir alta ganancia de información.
- ID3 clásico trabaja mejor con atributos categóricos.
- No incluye poda (*pruning*) en su formulación básica.
- Puede ser inestable: pequeños cambios en los datos pueden alterar la estructura del árbol.

## 8. Source code — implementación ID3 desde cero

El siguiente código implementa ID3 de forma didáctica usando un dataset clásico tipo **Play Tennis**.

In [ ]:
# ============================================================
# Instalación opcional para Colab / Anaconda
# ============================================================
# !pip install numpy pandas scikit-learn matplotlib

In [1]:
# ============================================================
# PASO 1: Importar librerías
# ============================================================
import numpy as np
import pandas as pd

print('Librerías importadas correctamente')

Librerías importadas correctamente


In [2]:
# ============================================================
# PASO 2: Crear dataset categórico Play Tennis
# ============================================================
data = pd.DataFrame([
    ['Sunny',    'Hot',  'High',   'Weak',   'No'],
    ['Sunny',    'Hot',  'High',   'Strong', 'No'],
    ['Overcast', 'Hot',  'High',   'Weak',   'Yes'],
    ['Rain',     'Mild', 'High',   'Weak',   'Yes'],
    ['Rain',     'Cool', 'Normal', 'Weak',   'Yes'],
    ['Rain',     'Cool', 'Normal', 'Strong', 'No'],
    ['Overcast', 'Cool', 'Normal', 'Strong', 'Yes'],
    ['Sunny',    'Mild', 'High',   'Weak',   'No'],
    ['Sunny',    'Cool', 'Normal', 'Weak',   'Yes'],
    ['Rain',     'Mild', 'Normal', 'Weak',   'Yes'],
    ['Sunny',    'Mild', 'Normal', 'Strong', 'Yes'],
    ['Overcast', 'Mild', 'High',   'Strong', 'Yes'],
    ['Overcast', 'Hot',  'Normal', 'Weak',   'Yes'],
    ['Rain',     'Mild', 'High',   'Strong', 'No']
], columns=['Outlook', 'Temperature', 'Humidity', 'Wind', 'Play'])

target = 'Play'
features = [col for col in data.columns if col != target]

data

,Outlook,Temperature,Humidity,Wind,Play
0,Sunny,Hot,High,Weak,No
1,Sunny,Hot,High,Strong,No
2,Overcast,Hot,High,Weak,Yes
3,Rain,Mild,High,Weak,Yes
4,Rain,Cool,Normal,Weak,Yes
5,Rain,Cool,Normal,Strong,No
6,Overcast,Cool,Normal,Strong,Yes
7,Sunny,Mild,High,Weak,No
8,Sunny,Cool,Normal,Weak,Yes
9,Rain,Mild,Normal,Weak,Yes


In [3]:
# ============================================================
# PASO 3: Definir entropía y ganancia de información
# ============================================================
def entropy(y: pd.Series) -> float:
    # Calcula la entropía de una variable categórica.
    probabilities = y.value_counts(normalize=True)
    return float(-np.sum(probabilities * np.log2(probabilities)))


def information_gain(df: pd.DataFrame, feature: str, target: str) -> float:
    # Calcula Information Gain para un atributo.
    parent_entropy = entropy(df[target])
    weighted_child_entropy = 0.0

    for _, subset in df.groupby(feature, dropna=False):
        weight = len(subset) / len(df)
        weighted_child_entropy += weight * entropy(subset[target])

    return parent_entropy - weighted_child_entropy


print('Entropía inicial:', round(entropy(data[target]), 4))

for feature in features:
    print(f'Information Gain({feature}):', round(information_gain(data, feature, target), 4))

Entropía inicial: 0.9403
Information Gain(Outlook): 0.2467
Information Gain(Temperature): 0.0292
Information Gain(Humidity): 0.1518
Information Gain(Wind): 0.0481


In [4]:
# ============================================================
# PASO 4: Implementar ID3 recursivo
# ============================================================
def majority_class(y: pd.Series):
    # Devuelve la clase mayoritaria.
    return y.value_counts().idxmax()


def id3(df: pd.DataFrame, target: str, features: list, default_class=None):
    # Construye un árbol ID3 usando diccionarios anidados.
    if len(df) == 0:
        return default_class

    # Si todas las muestras tienen la misma clase, crear hoja
    if df[target].nunique() == 1:
        return df[target].iloc[0]

    # Si no quedan atributos, usar clase mayoritaria
    if len(features) == 0:
        return majority_class(df[target])

    default_class = majority_class(df[target])

    # Seleccionar el atributo con mayor Information Gain
    gains = {feature: information_gain(df, feature, target) for feature in features}
    best_feature = max(gains, key=gains.get)

    tree = {best_feature: {}}
    remaining_features = [feature for feature in features if feature != best_feature]

    # Crear ramas por cada valor del atributo elegido
    for value, subset in df.groupby(best_feature, dropna=False):
        subtree = id3(subset, target, remaining_features, default_class)
        tree[best_feature][value] = subtree

    # Rama de respaldo para valores no vistos en entrenamiento
    tree[best_feature]['__default__'] = default_class
    return tree


id3_tree = id3(data, target, features)
id3_tree

{'Outlook': {'Overcast': 'Yes',
  'Rain': {'Wind': {'Strong': 'No', 'Weak': 'Yes', '__default__': 'Yes'}},
  'Sunny': {'Humidity': {'High': 'No', 'Normal': 'Yes', '__default__': 'No'}},
  '__default__': 'Yes'}}

In [5]:
# ============================================================
# PASO 5: Imprimir reglas del árbol
# ============================================================
def print_tree(tree, indent=''):
    # Imprime el árbol ID3 en formato legible.
    if not isinstance(tree, dict):
        print(indent + '=> Clase:', tree)
        return

    feature = next(iter(tree))
    branches = tree[feature]
    for value, subtree in branches.items():
        if value == '__default__':
            continue
        print(indent + f'SI {feature} == {value}:')
        print_tree(subtree, indent + '   ')


print_tree(id3_tree)

SI Outlook == Overcast:
   => Clase: Yes
SI Outlook == Rain:
   SI Wind == Strong:
      => Clase: No
   SI Wind == Weak:
      => Clase: Yes
SI Outlook == Sunny:
   SI Humidity == High:
      => Clase: No
   SI Humidity == Normal:
      => Clase: Yes


In [6]:
# ============================================================
# PASO 6: Predicción con ID3
# ============================================================
def predict_one(tree, sample: dict):
    # Predice una muestra con el árbol ID3.
    if not isinstance(tree, dict):
        return tree

    feature = next(iter(tree))
    branches = tree[feature]
    value = sample.get(feature)

    if value in branches:
        return predict_one(branches[value], sample)
    return branches.get('__default__')


predictions = [predict_one(id3_tree, row.to_dict()) for _, row in data[features].iterrows()]
accuracy = np.mean(data[target].values == np.array(predictions))

print('Predicciones:', predictions)
print('Accuracy sobre entrenamiento:', round(accuracy * 100, 2), '%')

new_sample = {
    'Outlook': 'Sunny',
    'Temperature': 'Cool',
    'Humidity': 'High',
    'Wind': 'Strong'
}

print('Nueva muestra:', new_sample)
print('Predicción ID3:', predict_one(id3_tree, new_sample))

Predicciones: ['No', 'No', 'Yes', 'Yes', 'Yes', 'No', 'Yes', 'No', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'No']
Accuracy sobre entrenamiento: 100.0 %
Nueva muestra: {'Outlook': 'Sunny', 'Temperature': 'Cool', 'Humidity': 'High', 'Wind': 'Strong'}
Predicción ID3: No


## 9. Uso práctico con scikit-learn

`scikit-learn` no implementa ID3 puro, pero `DecisionTreeClassifier(criterion='entropy')` permite entrenar un árbol de decisión con criterio de entropía, que reproduce la lógica principal de ID3 para seleccionar divisiones.

In [7]:
# ============================================================
# PASO 7: Árbol práctico con scikit-learn usando entropía
# ============================================================
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import accuracy_score, classification_report

X = data[features]
y = data[target]

# Compatibilidad entre versiones de scikit-learn
try:
    encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:
    encoder = OneHotEncoder(handle_unknown='ignore', sparse=False)

X_encoded = encoder.fit_transform(X)
encoded_feature_names = encoder.get_feature_names_out(features)

clf = DecisionTreeClassifier(
    criterion='entropy',
    random_state=42
)
clf.fit(X_encoded, y)

sk_predictions = clf.predict(X_encoded)
print('Accuracy sklearn:', round(accuracy_score(y, sk_predictions) * 100, 2), '%')
print(classification_report(y, sk_predictions))
print(export_text(clf, feature_names=list(encoded_feature_names)))

Accuracy sklearn: 100.0 %
              precision    recall  f1-score   support

          No       1.00      1.00      1.00         5
         Yes       1.00      1.00      1.00         9

    accuracy                           1.00        14
   macro avg       1.00      1.00      1.00        14
weighted avg       1.00      1.00      1.00        14

|--- Outlook_Overcast <= 0.50
|   |--- Humidity_Normal <= 0.50
|   |   |--- Outlook_Rain <= 0.50
|   |   |   |--- class: No
|   |   |--- Outlook_Rain >  0.50
|   |   |   |--- Wind_Strong <= 0.50
|   |   |   |   |--- class: Yes
|   |   |   |--- Wind_Strong >  0.50
|   |   |   |   |--- class: No
|   |--- Humidity_Normal >  0.50
|   |   |--- Wind_Strong <= 0.50
|   |   |   |--- class: Yes
|   |   |--- Wind_Strong >  0.50
|   |   |   |--- Outlook_Sunny <= 0.50
|   |   |   |   |--- class: No
|   |   |   |--- Outlook_Sunny >  0.50
|   |   |   |   |--- class: Yes
|--- Outlook_Overcast >  0.50
|   |--- class: Yes



In [8]:
# ============================================================
# PASO 8: Predicción de una nueva muestra con scikit-learn
# ============================================================
new_df = pd.DataFrame([new_sample])
new_encoded = encoder.transform(new_df)
print('Predicción sklearn:', clf.predict(new_encoded)[0])

Predicción sklearn: No


## 10. Conclusión

La implementación desarrollada reproduce fielmente la lógica de ID3 y alcanza una exactitud del 100 % sobre el conjunto de entrenamiento *Play Tennis*, generando además un árbol idéntico, en su criterio de división, al obtenido con `DecisionTreeClassifier(criterion='entropy')` de `scikit-learn`. Este contraste confirma la corrección de la formulación basada en entropía y ganancia de información.

La principal fortaleza de ID3 reside en su **interpretabilidad**: el modelo resultante se traduce de forma directa en reglas legibles que facilitan la explicación de las decisiones. Su limitación más relevante es la **propensión al sobreajuste** cuando el árbol crece sin control, así como el sesgo hacia atributos con muchos valores posibles. Por estas razones, en contextos productivos suelen preferirse sus sucesores —C4.5, C5.0 o CART—, que incorporan poda y un mejor tratamiento de variables numéricas y datos faltantes.